In [ ]:
import pandas as pd

books = pd.read_csv("../dataset/books_genre.csv")
books["content_features"] = books["content_features"].fillna("")
books["content_features"] = books["content_features"].astype(str)
print(books["content_features"].isnull().sum())


0


In [91]:
from sklearn.feature_extraction.text import TfidfVectorizer



tfidf = TfidfVectorizer( stop_words = "english" , max_features = 5000)


tfidf_matrix = tfidf.fit_transform(books["content_features"])
print(tfidf_matrix.shape)

(9818, 5000)


In [92]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_simi = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(cosine_simi.shape)

(9818, 9818)


In [93]:
def recommend_book(title, topn = 10):

    title = title.lower()
    
    books["title_clean"] = books["title"].fillna("").str.lower()

    matches = books[books["title_clean"].str.contains(title)]
    if matches.empty:
        return "Book not found"
    
    index = matches.index[0]
    simi_scores = list(enumerate(cosine_simi[index]))
    simi_scores = sorted(simi_scores, key=lambda x: x[1] , reverse=True )
    simi_scores = simi_scores[1:topn+1]
    indices = [i[0] for i in simi_scores]

    return books.iloc[indices][["title","authors","tag_name"]]

In [143]:
recommend_book("atonement")

,title,authors,tag_name
570,everything is illuminated,Jonathan Safran Foer,to-read fiction favorites historical-fiction b...
110,catch22,Joseph Heller,to-read currently-reading classics fiction fav...
541,the virgin suicides,Jeffrey Eugenides,to-read favorites fiction currently-reading yo...
6433,the good soldier,"Ford Madox Ford, Kenneth Womack, William Baker",to-read fiction classics currently-reading fav...
804,a room with a view,E.M. Forster,to-read classics fiction favorites romance cla...
1043,vanity fair,"William Makepeace Thackeray, John Carey",to-read classics currently-reading fiction cla...
5317,small island,Andrea Levy,to-read fiction currently-reading historical-f...
1643,lady chatterleys lover,"D.H. Lawrence, Doris Lessing, Chester Brown",classics to-read romance classic owned books-i...
1399,sophies choice,William Styron,to-read currently-reading fiction classics his...
2353,a passage to india,"E.M. Forster, Oliver Stallybrass, Pankaj Mishra",to-read classics currently-reading fiction ind...


In [95]:
books["title"].head(20)


0                   the hunger games the hunger games 1
1     harry potter and the sorcerers stone harry pot...
2                                   twilight twilight 1
3                                 to kill a mockingbird
4                                      the great gatsby
5                                the fault in our stars
6                                            the hobbit
7                                the catcher in the rye
8                      angels  demons  robert langdon 1
9                                   pride and prejudice
10                                      the kite runner
11                                divergent divergent 1
12                                                 1984
13                                          animal farm
14                            the diary of a young girl
15         the girl with the dragon tattoo millennium 1
16                     catching fire the hunger games 2
17    harry potter and the prisoner of azkaban h

In [99]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

cosin_simi = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [100]:
def hybrid_recommender(user_id, book_title, topn = 5):

    book_title = book_title.lower()
    books["title_clean"] = books["title"].str.lower()
    
    matches = books[books["title_clean"].str.contains(book_title, na=False)]

    if matches.empty:
        return "Book not found"

    index = matches.index[0] 
    simi_scores = list(enumerate(cosine_simi[index]))
    simi_scores = sorted(simi_scores, key=lambda x: x[1], reverse=True)
    recommendations = []

    for i , score in simi_scores:
        book_id = books.iloc[i]["book_id"]
        svd_score = svd_model.predict(user_id, book_id).est
        final_score = 0.6 * svd_score + 0.4 * score
        recommendations.append((books.iloc[i]["title"], final_score))

    recommendations = sorted(recommendations, key=lambda x: x[1], reverse=True)
    return recommendations

In [101]:
hybrid_recommender(user_id=1, book_title="Jane")

[('amsterdam', 2.8778603193221364),
 ('the curious incident of the dog in the nighttime', 2.8426855481245923),
 ('women in love brangwen family 2', 2.8390464645058118),
 ('hard times', 2.8271686831743184),
 ('tears of the giraffe no 1 ladies detective agency 2', 2.8264560909017757),
 ('still life with woodpecker', 2.808189184538185),
 ('girl with a pearl earring', 2.7978796104247965),
 ('villa incognito', 2.7764029875280016),
 ('peter and the shadow thieves peter and the starcatchers 2',
  2.7757730859952434),
 ('love in the time of cholera', 2.768282410405617),
 ('the taste of home cookbook', 2.7655347239564625),
 ('the beautiful and damned', 2.756070643816708),
 ('madame bovary', 2.7537126258118443),
 ('on beauty', 2.73364104918651),
 ('of mice and men', 2.728315157643904),
 ('atlas shrugged', 2.7282616006052507),
 ('treasure island', 2.72709564735362),
 ('a peoples history of the united states', 2.724481530363071),
 ('franny and zooey', 2.721899716585522),
 ('cane river', 2.71043747

In [102]:
import joblib

joblib.dump(tfidf, "../backend/models/tfidf_vectorizer.pkl")
joblib.dump(cosine_simi, "../backend/models/cosine_sim.pkl")


['../backend/models/cosine_sim.pkl']

In [108]:
books["popularity"] = books["average_rating"] * books["ratings_count"]

trending_books = books.sort_values(by= "popularity" , ascending=False).head(20)
trending_books[["title", "authors", "average_rating", "ratings_count"]]


,title,authors,average_rating,ratings_count
0,the hunger games the hunger games 1,Suzanne Collins,4.34,4780653
1,harry potter and the sorcerers stone harry pot...,"J.K. Rowling, Mary GrandPré",4.44,4602479
2,twilight twilight 1,Stephenie Meyer,3.57,3866839
3,to kill a mockingbird,Harper Lee,4.25,3198671
4,the great gatsby,F. Scott Fitzgerald,3.89,2683664
5,the fault in our stars,John Green,4.26,2346404
6,the hobbit,J.R.R. Tolkien,4.25,2071616
9,pride and prejudice,Jane Austen,4.24,2035490
17,harry potter and the prisoner of azkaban harry...,"J.K. Rowling, Mary GrandPré, Rufus Beck",4.53,1832823
12,1984,"George Orwell, Erich Fromm, Celâl Üster",4.14,1956832


In [129]:
def recommend_by_description(query, top_n=10):
    query_vec = tfidf.transform([query])
    sim_scores = cosine_similarity(query_vec, tfidf_matrix)
    
    scores = list(enumerate(sim_scores[0]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    book_indices = [i[0] for i in scores[:top_n]]
    
    return books.iloc[book_indices][["title", "authors", "tag_name"]]


In [139]:
recommend_by_description("A sad love story book")
recommend_by_description("fantasy book about children and animals")



,title,authors,tag_name
6167,wildwood wildwood chronicles 1,"Colin Meloy, Carson Ellis",to-read currently-reading fantasy young-adult ...
7779,the akhenaten adventure children of the lamp 1,P.B. Kerr,to-read fantasy young-adult currently-reading ...
8107,five children and it five children 1,E. Nesbit,to-read fantasy currently-reading classics chi...
6074,charmed life chrestomanci 1,Diana Wynne Jones,fantasy young-adult favorites ya fiction child...
8273,babar the king,Jean de Brunhoff,to-read childrens children children-s picture-...
467,the amber spyglass his dark materials 3,Philip Pullman,to-read fantasy young-adult favorites fiction ...
6949,the lives of christopher chant chrestomanci 2,Diana Wynne Jones,fantasy favorites to-read ya fiction childrens...
359,the subtle knife his dark materials 2,Philip Pullman,fantasy young-adult favorites to-read fiction ...
60,the golden compass his dark materials 1,Philip Pullman,to-read fantasy favorites young-adult currentl...
6360,the dark hills divide the land of elyon 1,Patrick Carman,fantasy to-read currently-reading favorites yo...


In [136]:
def recommend_by_genre(genre, topn=10):
    filter = books[books["tag_name"].str.contains(genre, case=False, na=False)]
    return filter.sort_values(by="ratings_count",ascending=False).head(topn)[["title","authors","tag_name"]]

In [144]:
recommend_by_genre("romance comedy")

,title,authors,tag_name
105,confessions of a shopaholic shopaholic 1,Sophie Kinsella,to-read favorites sophie-kinsella chic-lit fun...
616,ive got your number,Sophie Kinsella,to-read chicklit contemporary-romance funny re...
915,wallbanger cocktail 1,Alice Clayton,favorites currently-reading to-read contempora...
1093,2 states the story of my marriage,Chetan Bhagat,romance favorites indian-authors chetan-bhagat...
5330,second helpings jessica darling 2,Megan McCafferty,to-read young-adult favorites ya chick-lit fic...
7374,mars volume 01,Fuyumi Soryo,to-read manga romance favorites graphic-novels...
6456,yu yu hakusho volume 1 goodbye material world ...,Yoshihiro Togashi,to-read manga favorites fantasy graphic-novels...
6614,vinegar girl,Anne Tyler,to-read currently-reading fiction romance cont...
9551,millennium snow vol 1,Bisco Hatori,to-read manga romance vampires fantasy current...
9736,trouble in mudbug ghostinlaw 1,Jana Deleon,to-read currently-reading mystery kindle roman...


In [155]:
def recommend_by_author(name, topn=15):
    filter = books[books["authors"].str.contains(name, case=False, na=False)]
    return filter.sort_values(by="ratings_count", ascending=False).head(topn)[["title","authors"]]

In [158]:
recommend_by_author("agatha christie")


,title,authors
195,and then there were none,Agatha Christie
653,murder on the orient express hercule poirot 10,Agatha Christie
652,the mysterious affair at styles hercule poirot 1,"Agatha Christie, Ροζίτα Σώκου"
1028,murder at the vicarage miss marple 1,Agatha Christie
1374,the murder of roger ackroyd hercule poirot 4,Agatha Christie
1703,death on the nile hercule poirot 17,Agatha Christie
1694,the man in the brown suit,Agatha Christie
2189,the abc murders hercule poirot 13,Agatha Christie
3171,the body in the library miss marple 3,Agatha Christie
4160,evil under the sun hercule poirot 23,Agatha Christie
